## Scheduling data pipelines with Apache Airflow

* When building pipelines, some teams choose to create pipelines per layer (eg, bronze), per business unit, per frequency, etc.
* As your team and individual proficiency grow, teams move towards a 1 pipeline 1 output table pattern.
* Having only one table per pipeline makes it easy for data engineers to rerun pipelines when issues arise.
* We saw how to run our pipeline scripts with uv run. In a production system, we will need the following:
  * A schedule that controls when the pipeline is to be run
  * What date the pipeline should start running, and when it should stop running
  * If the pipeline was paused while you were fixing an issue and then unpaused, should it try to catch up on the runs during the paused time?
  * Name of the pipeline to be displayed on the UI
  * Are there any logic/code we want to execute when a pipeline completes or fails?
  * Are there more scripts to run than just our pipeline creation? If yes, what is the order of these?
  * etc
* To address these needs, Apache Airflow is a suitable tool.
* `Apache Airflow` provides an API that enables us to define the characteristics of our pipeline. It has a robust and wide range of capabilities.
* Airflow has specific concepts.
  * `DAG:` A directed acyclic graph, meaning that there are no infinite loops and the graph has a completion point. Airflow represents a pipeline as a DAG.
  * `Tasks:` A DAG consists of one or more tasks. The task represents the individual steps of data processing within a DAG. For example, a DAG may have a task to create a table and another to send an email.
  * `Task Dependencies` Airflow allows you to organize tasks in any way you want to (as long as there are no infinite loops). We can create branching logic, conditional task runs, etc.

#### Example

Let’s look at an example pipeline that creates the `local.bronze.customer` table.

First, we need an empty `__init__.py` file in this directory for our Airflow DAG to import the customer.py module.

Airflow frequently looks for new DAGs in the dags_folder at `$AIRFLOW_HOME/dags`. Let’s create a DAG in that location.

In [ ]:
! touch __init__.py

In [ ]:
! echo $AIRFLOW_HOME/dags

In [ ]:
# file: airflow/dags/bronze_customer.py
import time

import pendulum
from airflow.sdk import dag, task
from pyspark.sql import Row, SparkSession

from notebooks.bronze.customer import run as customer_run


@dag(
    dag_id="bronze.customer",
    schedule="@hourly",
    start_date=pendulum.datetime(2025, 1, 1, tz="UTC"),
    catchup=False,
    tags=["bronze"],
)
def bronze_customer_pipeline():

    @task(retries=2, retry_delay=2)
    def run_bronze_customer_etl() -> None:

        spark = (
            SparkSession.builder.appName("bronze.customer")
            .master("local[*]")
            .getOrCreate()
        )

        customer_run(spark)

    run_bronze_customer_etl()


bronze_customer_pipeline()

* In this example we use the decorator pattern to define DAG and tasks (aka TaskFlow API) 

* TaskFlow API was introduced in Airflow 2.0. You can also create DAGs with the older format that uses `with DAG`.
We use the Pythonic TaskFlow API, which lets us run Python functions as tasks and build DAGs.
* In the example we define the following for our DAG
  - dag_id: The ID that will be stored in Airflow metadata and show up in the UI
  - schedule: We can use any arbitrary cron schedule, and we can use [Airflow presets](https://airflow.apache.org/docs/apache-airflow/stable/authoring-and-scheduling/cron.html#cron-presets) (convenient English frequency). We used the `hourly` preset, which means our pipeline will run at the end of each hour.
  - start_date: Defines when our pipeline should start
  - catchup: If our pipeline is paused or starts in the past, do we need it to catch up on the runs that it did not run?
  - tags: Useful for filtering in the UI and grouping related DAGs
* For our task we defined the number of retries and how may seconds to wait between them. 

* Retries help when we are dealing with flaky code/systems.

* Open Airflow at [http://localhost:8080/](http://localhost:8080/)

#### Exercise [10 min]

Assume that our team decides that our bronze pipelines are similar enough that they do not need to be run individually.

Create an hourly pipeline called `bronze_layer` that runs all the bronze scripts in the `./bronze/` folder.

## Time range of data to be processed is supplied by Airflow

* The scheduler (Airflow) is responsible for providing the right start and end times to the pipeline script.
* This is often referred as time range, data interval, etc and Airflow has built in mechanism for this.

#### Example

Create a pipeline to run `local.silver.fct_orders` every hour.

In [ ]:
import pendulum
from airflow.sdk import dag, get_current_context, task
from airflow.timetables.interval import CronDataIntervalTimetable
from pyspark.sql import Row, SparkSession

from notebooks.silver.fct_orders import run as fct_orders_run


@dag(
    dag_id="silver.fct_orders",
    schedule=CronDataIntervalTimetable(
        "0 * * * *",  # 0th minute of every hour
        timezone=pendulum.timezone("UTC"),
    ),
    start_date=pendulum.datetime(2025, 1, 1, tz="UTC"),
    catchup=False,
    tags=["silver", "orders"],
)
def silver_fct_orders_pipeline():
    @task(retries=2, retry_delay=2)
    def run_silver_fct_orders_etl() -> None:
        context = get_current_context()

        start_time = context["data_interval_start"].strftime("%Y-%m-%d %H:%M:%S")
        end_time = context["data_interval_end"].strftime("%Y-%m-%d %H:%M:%S")
        spark = (
            SparkSession.builder.appName("silver_fct_orders")
            .master("local[*]")
            .getOrCreate()
        )
        print(f"Starting pipeline for time range {start_time} to {end_time}")

        fct_orders_run(spark, start_time, end_time)

    run_silver_fct_orders_etl()


silver_fct_orders_pipeline()

* We use the `CronDataIntervalTimetable` as it is meant for data interval type patterns. I

* We get the start and end time from a `context` object.
* The `context` object which we get by calling `get_current_context` has information about the current run of the DAG.

#### Exercise [10 min]

Create a pipeline to run `local.silver.fct_order_lines` on the 1st of every month. Make sure it catches up to `2025-01-01`, since our input data is for 2025.

Use [crontab guru](https://crontab.guru/#0_0_1_*_*) to get the cron schedule.


* **Lambda Architecture:**

  * The fast path runs at a shorter frequency. 
  * slow path at a longer frequency, but with a wider time range of input.

![Lambda Architecture](images/lambda_architecture.png)

* We can create this slow path using the lookback pattern.

#### Example

Let's consider a simple example where our pipeline runs daily, but data may arrive up to 6 days late.

We can use a 7-day lookback to catch any late-arriving data, as shown below.

In [ ]:
from IPython.display import IFrame

IFrame(src="lookback.html", width="100%", height="600")

#### Exercise [15 min]

Create a DAG called `fast_and_slow_tasks` that runs every minute with 2 tasks.
  - fast_task runs every minute and prints its start and end time
  - slow_task runs every 5th minute (0, 5, 10, 15, ...) of the hour with 1 hour lookback and prints its start and end time

*Note* fast_task should not run on the 5th minute that the slow_task is running.

*Ref* Use the `@task.branch` decorator to implement this. [Documentation](https://airflow.apache.org/docs/apache-airflow/stable/core-concepts/dags.html#branching)

* Turn on the DAG and wait for it run atleat 6 times to verify the result



## Running pipeline when a dataset is updated

* As your team and responsibilities grow, it will get increasingly difficult to figure out when to run which pipelines.
* Typically teams use best guess estimate to schedule pipelines.
* In this approach the team will estimate that a certain upstream pipeline may finish at n hour and the pipeline underconsideration will be started at n + (30m/1h) hour.
* This approach is fragile a small upstream delay can cause the downstream pipelines to use old data.
* Teams workaround this limitation in various ways. Some of which are
  - Using tasks to constantly check that the upstream data is ready for use
  - Re-running pipelines multiple times, assuming one of them would pick up the latest upstream
* In recent years, scheduler (eg. Airflow, Dagster) have begun providing data update based triggering of pipelines.
* In this approach the upstream pipelines will tell the scheduler that a data has been updated. 

* The scheduler will identify any downstream pipelines that depend on this upstream and start them.
* With this approach teams, do not need to guess when upstream data will be ready and use a deterministic way to trigger downstream pipelines.

#### Example

Let's create a pipeline `fct_order_lines_et` which gets triggered when the upstream `bronze.order_lines` gets updated.

**Note** An update can refer to any operation in this case, as seen in the code below

In [ ]:
# Create a new DAG ./airflow/dags/bronze_order_lines.py
# file: airflow/dags/bronze_customer.py
# file: airflow/dags/bronze_customer.py
import time

import pendulum
from airflow.sdk import Asset, Metadata, dag, get_current_context, task
from airflow.timetables.interval import CronDataIntervalTimetable
from pyspark.sql import Row, SparkSession

from notebooks.bronze.order_lines import run as order_lines_run

bronze_order_lines = Asset(
    "file://opt/spark/warehouse/local/bronze/order_lines/part-*.parquet"
)  # Defines bronze.order_lines as a data asset


@dag(
    dag_id="bronze.order_lines",
    schedule=CronDataIntervalTimetable(
        "0 * * * *",  # 0th minute of every hour
        timezone=pendulum.timezone("UTC"),
    ),
    start_date=pendulum.datetime(2025, 1, 1, tz="UTC"),
    catchup=False,
    tags=["bronze", "event-driven"],
)
def bronze_order_lines_pipeline():
    @task(
        outlets=[bronze_order_lines]
    )  # <-- marks the asset as updated on task success
    def run_bronze_order_lines_etl() -> None:

        spark = (
            SparkSession.builder.appName("bronze.order_lines")
            .master("local[*]")
            .getOrCreate()
        )

        order_lines_run(spark)
        # getting start and end time to send downstream to act as its time range
        context = get_current_context()

        start_time = context["data_interval_start"].strftime("%Y-%m-%d %H:%M:%S")
        end_time = context["data_interval_end"].strftime("%Y-%m-%d %H:%M:%S")
        yield Metadata(
            bronze_order_lines, {"start_time": start_time, "end_time": end_time}
        )

    run_bronze_order_lines_etl()


bronze_order_lines_pipeline()

In [ ]:
# Create a downstream DAG that depends on the updates to
# data asset defined above
# airflow/dags/silver_fct_order_lines_et.py
import pendulum
from airflow.sdk import Asset, dag, get_current_context, task
from pyspark.sql import Row, SparkSession

from notebooks.silver.fct_order_lines import run as fct_order_lines_run

bronze_order_lines = Asset(
    "file://opt/spark/warehouse/local/bronze/order_lines/part-*.parquet"
)  # Defines bronze.order_lines as a data asset


@dag(
    dag_id="silver.fct_order_lines_et",
    schedule=[bronze_order_lines],  # <-- triggered by asset updates, not by time
    start_date=pendulum.datetime(2025, 1, 1, tz="UTC"),
    catchup=True,
    max_active_runs=1,  # only one run at a time, in order
    tags=["silver", "order_lines", "event-driven"],
)
def silver_fct_order_lines_pipeline():
    @task(retries=2, retry_delay=2, inlets=[bronze_order_lines])
    def run_silver_fct_order_lines_etl() -> None:

        spark = (
            SparkSession.builder.appName("silver_fct_orders")
            .master("local[*]")
            .getOrCreate()
        )

        # Get start and end times from the upstream's metadata
        context = get_current_context()
        inlet_events = context["inlet_events"][bronze_order_lines]
        latest_event = inlet_events[-1]
        start_time = latest_event.extra.get("start_time")
        end_time = latest_event.extra.get("end_time")
        print(f"Starting pipeline for time range {start_time} to {end_time}")

        fct_order_lines_run(spark, start_time, end_time)

    run_silver_fct_order_lines_etl()


silver_fct_order_lines_pipeline()

#### Exercise [15 min]

Create the pipeline `fct_order_lines_multi_upstream_et.py` which depends on both
1. The order_lines data asset we defined above
2. A new warehouse data asset defined in a new`bronze_warehouse.py` pipeline script (hourly runs). Assume we just need to wait for this data asset to start the processing of `silver.fct_order_lines`.

*Note* The start and end times must come only from bronze.order_lines

## Airflow Architecture

* Airflow is used to refer to the Airflow library (e.g. `from airflow.sdk import Asset`) and the Airflow system.
* The Airflow system has 5 main parts
  1. **Dag processor** is a Python process that monitors the DAGs folder. When it identifies a Python file with a DAG, it serializes the script and stores it in a database
  2. **Scheduler** is a Python process that periodically checks the database for pipelines to run. This will enforce the parameters set at the DAG level (e.g., consecutive running DAGs)
  3. **Executor** is a system that is responsible for running tasks. We can define multiple types of executors, such as Sequential, Local, Celery, k8s, and custom. We use `LocalExecutor`, where each task runs as an individual Python process on our machine. The celery and k8s executors enable us to run our tasks using those systems.
  4. **UI server** is a flask app that reads data from the database and shows list of DAGs, their run history, variables, connections, etc It is used to interact with DAGs and configurations.
  5. **Database** Stores all the configuration information and historical information of all the DAG runs. All the information displayed in the UI comes from this database.

![Airflow Architecture](images/airflow_arch_basic.png)

### Example

Let's look at the running DAG processor and scheduler.

In [ ]:
! ps aux | grep '[d]ag-processor'

In [ ]:
! ps aux | grep '[a]irflow scheduler'

In [ ]:
! ps aux | grep '[a]irflow worker' | wc -l

* Our configuration information is stored in a text file called `airflow.cfg`.
* This file is typically in our `$AIRFLOW_HOME` path

In [ ]:
! cat $AIRFLOW_HOME/airflow.cfg | grep 'max_active_tasks_per_dag ='

In [ ]:
! cat $AIRFLOW_HOME/airflow.cfg | grep 'max_active_runs_per_dag ='

In [ ]:
! cat $AIRFLOW_HOME/airflow.cfg | grep 'parallelism ='

## Recap
In this section, we learnt how to schedule and orchestrate pipelines with Airflow. We saw how to 

1. Keep pipelines easy to maintain with 1 dag 1 output pattern
2. Use time ranges to process incremental data
3. Trigger pipelines based on updates to upstream data
4. See Airflow processes and how it works under the hood